In [5]:
import os
import yaml
import logging
from pyspark.sql.functions import rand, col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

from utils.spark_session import get_spark_session
from utils.training_utils import find_specific_variables

# Initialize Spark session
spark = get_spark_session(app_name="05-model-evaluation")

In [6]:
# Paths
feature_config_path = os.path.join('..', 'src', 'config', 'feature_config.yaml')
features_selected_path = os.path.join('..', 'src', 'features', 'selected', 'features_selected.yaml')
train_data_path = os.path.join('..', 'data', 'train_test', 'train.parquet')

In [7]:
# Load configuration
with open(feature_config_path, 'r') as f:
    feature_config = yaml.safe_load(f)
target_col = find_specific_variables(feature_config, 'target', specific_value=True)
target_col = target_col[0] if isinstance(target_col, list) else target_col

with open(features_selected_path, 'r') as f:
    selected_yaml = yaml.safe_load(f)
feature_cols = selected_yaml.get("support_random_forest", [])
if "index" in feature_cols:
    feature_cols.remove("index")

In [8]:
# Load training data
train_df = spark.read.parquet(train_data_path)
train_df.show(5)

+--------------------+---------------------+------+------------+---------------------+-----------+----------------+---+-----------------+------+-------------+-------+---------+--------+-----------+---------------------+---------------------+------------------+--------------------+----------------------------+--------------------+------------------+
|          account_id|time_since_test_start|amount|max_discount|completed_offer_types|real_amount|target_converted|age|credit_card_limit|gender|registered_on|reg_day|reg_month|reg_year|      index|real_amount_per_limit|n_transactions_so_far| amount_cumulative|n_conversions_so_far|pct_current_vs_total_session|    amount_pct_limit|limit_factor_vs_tx|
+--------------------+---------------------+------+------------+---------------------+-----------+----------------+---+-----------------+------+-------------+-------+---------+--------+-----------+---------------------+---------------------+------------------+--------------------+-----------------

In [9]:
# Create stratified folds (grouped by account_id)
num_folds = 5
account_folds = (
    train_df
    .select("account_id")
    .distinct()
    .withColumn("fold", (rand(seed=96) * num_folds).cast("int"))
)
df_with_folds = train_df.join(account_folds, on="account_id", how="left")


In [10]:
# Define classifiers
models = {
    'DT': DecisionTreeClassifier(labelCol="label", featuresCol="features"),
    'RF': RandomForestClassifier(labelCol="label", featuresCol="features", seed=96),
    'GBT': GBTClassifier(labelCol="label", featuresCol="features", seed=96)
}

In [11]:
# Define evaluation metric
evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

In [12]:
# Function to apply manual undersampling
def apply_manual_sampling(df, label_col="label", minority_class=1):
    df_pos = df.filter(col(label_col) == minority_class)
    df_neg = df.filter(col(label_col) != minority_class)
    neg_fraction = df_pos.count() / df_neg.count()
    df_neg_sampled = df_neg.sample(withReplacement=False, fraction=neg_fraction, seed=96)
    return df_pos.union(df_neg_sampled)

In [13]:
# Dictionary to store results
final_results = {"original": {}, "manual_sampling": {}}

# Evaluation: original vs. balanced
for setting in ["original", "manual_sampling"]:
    print(f"\n===== EVALUATION: {setting.upper()} =====")

    for model_name, classifier in models.items():
        print(f"\n---- Model: {model_name} ----")
        auc_scores = []

        for fold in range(num_folds):
            print(f"Fold {fold + 1}/{num_folds}")

            # Train-validation split (by fold)
            train_fold = df_with_folds.filter(col("fold") != fold)
            valid_fold = df_with_folds.filter(col("fold") == fold)

            # Assemble features
            assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
            train_vec = assembler.transform(train_fold).withColumnRenamed(target_col, "label").select("features", "label")
            valid_vec = assembler.transform(valid_fold).withColumnRenamed(target_col, "label").select("features", "label")

            # Apply manual sampling if needed
            if setting == "manual_sampling":
                train_vec = apply_manual_sampling(train_vec, label_col="label")

            # Fit model and evaluate
            model = classifier.fit(train_vec)
            predictions = model.transform(valid_vec)
            auc = evaluator.evaluate(predictions)
            auc_scores.append(auc)
            print(f"AUC: {auc:.4f}")

        mean_auc = sum(auc_scores) / num_folds
        final_results[setting][model_name] = mean_auc
        print(f"Mean AUC: {mean_auc:.4f}")


===== EVALUATION: ORIGINAL =====

---- Model: DT ----
Fold 1/5
AUC: 0.5717
Fold 2/5
AUC: 0.6105
Fold 3/5
AUC: 0.5878
Fold 4/5
AUC: 0.6876
Fold 5/5
AUC: 0.6942
Mean AUC: 0.6304

---- Model: RF ----
Fold 1/5
AUC: 0.8660
Fold 2/5
AUC: 0.8667
Fold 3/5
AUC: 0.8673
Fold 4/5
AUC: 0.8667
Fold 5/5
AUC: 0.8685
Mean AUC: 0.8670

---- Model: GBT ----
Fold 1/5
AUC: 0.9029
Fold 2/5
AUC: 0.9085
Fold 3/5
AUC: 0.9047
Fold 4/5
AUC: 0.9040
Fold 5/5
AUC: 0.9083
Mean AUC: 0.9057

===== EVALUATION: MANUAL_SAMPLING =====

---- Model: DT ----
Fold 1/5
AUC: 0.7348
Fold 2/5
AUC: 0.7130
Fold 3/5
AUC: 0.8438
Fold 4/5
AUC: 0.7491
Fold 5/5
AUC: 0.8555
Mean AUC: 0.7792

---- Model: RF ----
Fold 1/5
AUC: 0.8771
Fold 2/5
AUC: 0.8750
Fold 3/5
AUC: 0.8777
Fold 4/5
AUC: 0.8780
Fold 5/5
AUC: 0.8800
Mean AUC: 0.8775

---- Model: GBT ----
Fold 1/5
AUC: 0.9017
Fold 2/5
AUC: 0.9053
Fold 3/5
AUC: 0.9024
Fold 4/5
AUC: 0.9049
Fold 5/5
AUC: 0.9058
Mean AUC: 0.9040


In [14]:
# Print summary
print("\n===== FINAL RESULTS SUMMARY =====")
for setting in final_results:
    print(f"\nSetting: {setting}")
    for model_name, score in final_results[setting].items():
        print(f"{model_name}: AUC = {score:.4f}")


===== FINAL RESULTS SUMMARY =====

Setting: original
DT: AUC = 0.6304
RF: AUC = 0.8670
GBT: AUC = 0.9057

Setting: manual_sampling
DT: AUC = 0.7792
RF: AUC = 0.8775
GBT: AUC = 0.9040


In [15]:
spark.stop()